# ♟️ PWN3 vs Stockfish — jusqu'où monte PWN@ab3 ?

**PWN@ab3** (poids SPAWN + alpha-bêta profondeur 3 + quiescence) affronte **Stockfish 17 à plusieurs niveaux**,
du plus faible au plus fort. Chaque niveau joue les mêmes **ouvertures**, une fois avec chaque couleur.

On obtient :
- le **score contre chaque niveau** (la courbe doit descendre de ~100 % à ~0 %) ;
- l'**Elo en partie** de PWN@ab3 (maximum de vraisemblance, ancré sur les `UCI_Elo` de Stockfish) avec un intervalle à 95 % ;
- toutes les parties en **PGN** + **toutes les positions** (FEN après chaque coup) en CSV.

Pendant que ça tourne : les **échiquiers des parties en cours en direct**, le **tableau des scores** par niveau,
une ligne par partie terminée (qui a gagné), et un **bilan dès qu'un niveau est fini**.
Les fichiers de chaque partie sont écrits au fur et à mesure dans `pwn3_parties/stockfish_eloXXXX/`.

Repères : PWN@ab2 a fait **1904** en parties (notebook 03), PWN@ab3 **2345** aux puzzles (notebook 02).

⚠️ L'`UCI_Elo` de Stockfish est calibré contre des moteurs (échelle CCRL) : c'est un ordre de grandeur, pas un Elo Lichess ou FIDE.

⚙️ Kaggle : **Internet ON**. L'alpha-bêta tourne sur **CPU**, le GPU n'est pas nécessaire.
Durée : ~4–6 h (ab3 réfléchit ~5–15 s par coup) → lance-le en **Save & Run All (Commit)**.
Arrêt propre après `BUDGET_H` heures : les parties jouées sont gardées et analysées.

In [ ]:
# onnxruntime-gpu remplace onnxruntime (les deux ne cohabitent pas) ; marche aussi sans GPU.
# Version fixée : les plus récentes demandent CUDA 13, Kaggle a CUDA 12. [cuda,cudnn] installe les bibliothèques CUDA 12.
!pip uninstall -y -q onnxruntime onnxruntime-gpu > /dev/null 2>&1
!pip install -q chess zstandard "onnxruntime-gpu[cuda,cudnn]==1.24.1"

In [ ]:
%%writefile commun_echecs.py
"""Code commun aux notebooks de benchmark échecs (PAWN, PAWN big, contrôle, SPAWN/PWN).

Tout vient de pawn_app.py : même encodage 69 octets, même recherche « arbre complet »,
même alpha-bêta + quiescence. Seule différence : pas de bruit ni de coups au hasard,
on veut mesurer la force réelle.
"""
import glob
import json
import os
import shutil
import stat
import subprocess
import tarfile
import time
import urllib.request

import chess
import chess.engine
import chess.polyglot
import numpy as np
import onnxruntime as ort

DEPOT = "LiamLitle/ia-de-liam"
BRANCHE = "main"
LFS = f"https://media.githubusercontent.com/media/{DEPOT}/{BRANCHE}/"
DOSSIER_POIDS = os.environ.get("PAWN_POIDS", "/kaggle/working/poids" if os.path.isdir("/kaggle") else "poids")
MATE = 100_000

# nom -> chemin dans le dépôt
RESEAUX = {
    "PAWN": "Pawn/pawn.onnx",
    "PAWN_BIG": "PawnBig-V1/pawn_big.onnx",
    "CONTROLE": "PawnBig-controlleur/pawn_big_controle.onnx",
    "SPAWN": "PWN-soluce/pawn_soluce.onnx",
}


def poids(chemin_depot):
    """cherche le fichier dans /kaggle/input (dataset ajouté au notebook), sinon le télécharge depuis GitHub LFS"""
    nom = os.path.basename(chemin_depot)
    for racine in ("/kaggle/input", DOSSIER_POIDS):
        trouves = glob.glob(os.path.join(racine, "**", nom), recursive=True)
        trouves = [t for t in trouves if os.path.getsize(t) > 1000]  # pas un pointeur LFS
        if trouves:
            return trouves[0]
    os.makedirs(DOSSIER_POIDS, exist_ok=True)
    dest = os.path.join(DOSSIER_POIDS, nom)
    print(f"téléchargement de {chemin_depot} ...")
    urllib.request.urlretrieve(LFS + chemin_depot, dest)
    if os.path.getsize(dest) < 1000:
        raise RuntimeError(f"{dest} ressemble à un pointeur LFS, pas au vrai fichier")
    return dest


GPU_ID = 0       # carte utilisée par ce processus (Kaggle « GPU T4 x2 » en a deux)
GPU_MEM_MO = 0   # plafond mémoire GPU par réseau (0 = pas de plafond) ; fixé quand plusieurs processus partagent le GPU
_CUDA_OK = None  # None = pas encore essayé, False = échec -> on reste sur CPU sans réessayer


def providers(gpu=True):
    dispo = ort.get_available_providers()
    if gpu and _CUDA_OK is not False and "CUDAExecutionProvider" in dispo:
        try:
            ort.preload_dlls()  # charge CUDA/cuDNN depuis les paquets pip nvidia-* (déjà là avec torch sur Kaggle)
        except Exception:
            pass
        # HEURISTIC : cuDNN choisit ses algos sans essayer ceux qui demandent d'énormes
        # zones de travail (sinon des blocs de 150 Mo dépassent le plafond mémoire)
        opts = {"arena_extend_strategy": "kSameAsRequested", "cudnn_conv_algo_search": "HEURISTIC",
                "cudnn_conv_use_max_workspace": "0", "device_id": GPU_ID}
        if GPU_MEM_MO:
            opts["gpu_mem_limit"] = GPU_MEM_MO * 1024 * 1024
        return [("CUDAExecutionProvider", opts), "CPUExecutionProvider"]
    return ["CPUExecutionProvider"]


PIECES_ID = {c: i + 1 for i, c in enumerate("PNBRQK")}


def enc_fen(fen):
    f = fen.split(" ")
    noir = f[1] == "b"
    out = bytearray(69)
    r = c = 0
    for ch in f[0]:
        if ch == "/":
            r += 1
            c = 0
        elif ch.isdigit():
            c += int(ch)
        else:
            nous = ch.isupper() != noir
            out[(r if noir else 7 - r) * 8 + c] = PIECES_ID[ch.upper()] + (0 if nous else 6)
            c += 1
    ours, theirs = ("kq", "KQ") if noir else ("KQ", "kq")
    out[64] = ours[0] in f[2]
    out[65] = ours[1] in f[2]
    out[66] = theirs[0] in f[2]
    out[67] = theirs[1] in f[2]
    if f[3] != "-":
        out[68] = ord(f[3][0]) - 96
    return bytes(out)


class Evaluateur:
    """un réseau ONNX : liste de FEN -> centipions du point de vue du camp au trait"""

    def __init__(self, nom, gpu=True, threads=0):
        self.nom = nom
        self.so = ort.SessionOptions()
        if threads:
            self.so.intra_op_num_threads = threads
            self.so.inter_op_num_threads = 1
        global _CUDA_OK
        prov = providers(gpu)
        self.session = ort.InferenceSession(poids(RESEAUX[nom]), self.so, providers=prov)
        self.sur_gpu = "CUDAExecutionProvider" in self.session.get_providers()
        if prov[0] != "CPUExecutionProvider":
            _CUDA_OK = self.sur_gpu
            if not _CUDA_OK:
                print("⚠️ GPU indisponible pour onnxruntime : on continue sur CPU (plus lent mais résultats identiques)")
                ort.set_default_logger_severity(4)
        self.nb_evals = 0
        self.cache = {}

    def evals(self, fens, lot=4096):
        if not fens:
            return np.zeros(0, dtype=np.float32)
        if self.sur_gpu and GPU_MEM_MO:
            lot = min(lot, 512)  # petits lots quand la mémoire GPU est plafonnée
        res = []
        for i in range(0, len(fens), lot):
            x = np.frombuffer(b"".join(map(enc_fen, fens[i:i + lot])), np.uint8).reshape(-1, 69).copy()
            n = len(x)
            if self.sur_gpu and GPU_MEM_MO:
                # taille de lot arrondie à une puissance de 2 : peu de formes différentes, donc
                # onnxruntime réutilise ses blocs mémoire au lieu d'en allouer de nouveaux à chaque coup
                taille = max(64, 1 << (n - 1).bit_length())
                x = np.concatenate([x, np.zeros((taille - n, 69), np.uint8)]) if taille > n else x
            try:
                res.append(self.session.run(None, {"board": x})[0][:n])
            except Exception as e:
                if not self.sur_gpu:
                    raise
                # GPU plein : ce réseau passe sur CPU pour la suite (mêmes résultats, plus lent)
                print(f"⚠️ {self.nom} : erreur GPU ({str(e)[:80]}...) -> CPU")
                self.session = ort.InferenceSession(poids(RESEAUX[self.nom]), self.so, providers=["CPUExecutionProvider"])
                self.sur_gpu = False
                res.append(self.session.run(None, {"board": x})[0])
        self.nb_evals += len(fens)
        return np.concatenate(res)

    def eval1(self, fen):
        # l'alpha-bêta réévalue souvent les mêmes positions en quiescence : le réseau est
        # déterministe, donc un cache ne change rien au résultat, juste la vitesse
        v = self.cache.get(fen)
        if v is None:
            if len(self.cache) > 2_000_000:
                self.cache.clear()
            v = self.cache[fen] = float(self.evals([fen])[0])
        return v


# -- recherche « arbre complet » (PAWN, PAWN big, contrôle, SPAWN) -----------------

def arbre(board, depth, ply, feuilles):
    if not any(board.legal_moves):
        return ("t", -(MATE - ply) if board.is_check() else 0)
    if ply > 0 and (board.is_insufficient_material() or board.is_repetition(2) or board.halfmove_clock >= 100):
        return ("t", 0)
    if depth == 0:
        feuilles.append(board.fen())
        return ("f", len(feuilles) - 1)
    fils = []
    for mv in list(board.legal_moves):
        board.push(mv)
        fils.append((mv, arbre(board, depth - 1, ply + 1, feuilles)))
        board.pop()
    return ("n", fils)


def valeur(noeud, v):
    genre, x = noeud
    if genre == "t":
        return x
    if genre == "f":
        return v[x]
    return max(-valeur(c, v) for _, c in x)


def choisir_arbre(board, ev, depth):
    feuilles = []
    racine = arbre(board, depth, 0, feuilles)
    v = ev.evals(feuilles) if feuilles else []
    notes = [(-valeur(c, v), mv) for mv, c in racine[1]]
    return max(notes, key=lambda t: t[0])[1]


# -- recherche alpha-bêta + quiescence (PWN) --------------------------------------

VAL_PIECE = {chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3, chess.ROOK: 5, chess.QUEEN: 9, chess.KING: 0}


def trier_coups(board, coups, priorite):
    def cle(mv):
        if mv == priorite:
            return 10_000
        if board.is_capture(mv):
            prise = board.piece_type_at(mv.to_square) or chess.PAWN
            attaquant = board.piece_type_at(mv.from_square)
            return 1000 + VAL_PIECE[prise] * 10 - VAL_PIECE[attaquant]
        if board.gives_check(mv):
            return 500
        return 0
    return sorted(coups, key=cle, reverse=True)


def quiescence(board, ev, alpha, beta, prof_q):
    stand_pat = ev.eval1(board.fen())
    if stand_pat >= beta:
        return beta
    alpha = max(alpha, stand_pat)
    if prof_q <= 0:
        return alpha
    captures = [m for m in board.legal_moves if board.is_capture(m)]
    for mv in trier_coups(board, captures, None):
        board.push(mv)
        score = -quiescence(board, ev, -beta, -alpha, prof_q - 1)
        board.pop()
        if score >= beta:
            return beta
        alpha = max(alpha, score)
    return alpha


def negamax(board, ev, profondeur, alpha, beta, ply, prof_q, tt, coup_tt):
    alpha0 = alpha
    if not any(board.legal_moves):
        return -(MATE - ply) if board.is_check() else 0
    if ply > 0 and (board.is_insufficient_material() or board.is_repetition(2) or board.halfmove_clock >= 100):
        return 0
    hash_ = chess.polyglot.zobrist_hash(board)
    entree = tt.get(hash_)
    if entree and entree[0] >= profondeur:
        d, v, drapeau, _ = entree
        if drapeau == "exact":
            return v
        if drapeau == "min":
            alpha = max(alpha, v)
        elif drapeau == "max":
            beta = min(beta, v)
        if alpha >= beta:
            return v
    if profondeur <= 0:
        return quiescence(board, ev, alpha, beta, prof_q)
    meilleur, meilleur_coup = -MATE - 1, None
    for mv in trier_coups(board, list(board.legal_moves), coup_tt.get(hash_)):
        board.push(mv)
        score = -negamax(board, ev, profondeur - 1, -beta, -alpha, ply + 1, prof_q, tt, coup_tt)
        board.pop()
        if score > meilleur:
            meilleur, meilleur_coup = score, mv
        alpha = max(alpha, score)
        if alpha >= beta:
            break
    drapeau = "exact" if alpha0 < meilleur < beta else ("min" if meilleur >= beta else "max")
    tt[hash_] = (profondeur, meilleur, drapeau, meilleur_coup)
    if meilleur_coup is not None:
        coup_tt[hash_] = meilleur_coup
    return meilleur


def choisir_alphabeta(board, ev, depth, prof_q=4):
    tt, coup_tt = {}, {}
    meilleur_coup = None
    for d in range(1, depth + 1):
        alpha, beta = -MATE - 1, MATE + 1
        hash_ = chess.polyglot.zobrist_hash(board)
        meilleur_score, meilleur_du_tour = -MATE - 1, None
        for mv in trier_coups(board, list(board.legal_moves), coup_tt.get(hash_)):
            board.push(mv)
            score = -negamax(board, ev, d - 1, -beta, -alpha, 1, prof_q, tt, coup_tt)
            board.pop()
            if score > meilleur_score:
                meilleur_score, meilleur_du_tour = score, mv
            alpha = max(alpha, score)
        meilleur_coup = meilleur_du_tour
        coup_tt[hash_] = meilleur_coup
    return meilleur_coup


# -- joueurs ------------------------------------------------------------------------
# un « joueur » = un réseau + un algorithme de recherche + une profondeur.
# ex : "PAWN_BIG@arbre2", "SPAWN@arbre1", "PWN@ab3" (PWN = poids SPAWN + alpha-bêta).

def decoder(joueur):
    nom, rech = joueur.split("@")
    if nom == "PWN":
        nom = "SPAWN"
    if rech.startswith("arbre"):
        return nom, "arbre", int(rech[5:])
    if rech.startswith("ab"):
        return nom, "ab", int(rech[2:])
    raise ValueError(joueur)


_EVALS = {}
_SF = {}
GPU = True
THREADS = 0
SF_CHEMIN = None
SF_TEMPS = 0.1


def evaluateur(nom, gpu=None):
    gpu = GPU if gpu is None else gpu
    if (nom, gpu) not in _EVALS:
        _EVALS[(nom, gpu)] = Evaluateur(nom, gpu=gpu, threads=THREADS)
    return _EVALS[(nom, gpu)]


def configurer(gpu=True, threads=0, sf_chemin=None, sf_temps=0.1):
    """à appeler dans chaque processus fils avant de jouer"""
    global GPU, THREADS, SF_CHEMIN, SF_TEMPS
    GPU, THREADS, SF_CHEMIN, SF_TEMPS = gpu, threads, sf_chemin, sf_temps
    _EVALS.clear()
    _SF.clear()


def stockfish(cfg):
    """cfg = "elo1500" (UCI_LimitStrength) ou "skill0" (Skill Level)"""
    if cfg not in _SF:
        e = chess.engine.SimpleEngine.popen_uci(SF_CHEMIN)
        opts = {"Threads": 1, "Hash": 16}
        if cfg.startswith("elo"):
            opts.update({"UCI_LimitStrength": True, "UCI_Elo": int(cfg[3:])})
        elif cfg.startswith("skill"):
            opts["Skill Level"] = int(cfg[5:])
        e.configure(opts)
        _SF[cfg] = e
    return _SF[cfg]


def fermer_stockfish():
    for e in _SF.values():
        e.quit()
    _SF.clear()


def coup(joueur, board):
    if joueur.startswith("SF@"):
        return stockfish(joueur[3:]).play(board, chess.engine.Limit(time=SF_TEMPS)).move
    nom, rech, depth = decoder(joueur)
    b = board.copy(stack=True)
    if rech == "arbre":
        return choisir_arbre(b, evaluateur(nom), depth)
    # l'alpha-bêta évalue une position à la fois : sur Kaggle la latence CPU (~6 ms)
    # est deux fois plus basse que celle du GPU (~14 ms), donc on reste sur CPU
    return choisir_alphabeta(b, evaluateur(nom, gpu=False), depth)


# -- Stockfish -------------------------------------------------------------------------

URLS_STOCKFISH = [
    "https://github.com/official-stockfish/Stockfish/releases/download/sf_17.1/stockfish-ubuntu-x86-64-avx2.tar",
    "https://github.com/official-stockfish/Stockfish/releases/download/sf_17/stockfish-ubuntu-x86-64-avx2.tar",
    "https://github.com/official-stockfish/Stockfish/releases/download/sf_16.1/stockfish-ubuntu-x86-64-avx2.tar",
]


def rendre_executable(binaire):
    """/kaggle/working interdit d'exécuter des programmes : on copie Stockfish ailleurs
    et on garde le premier emplacement où il démarre vraiment"""
    for dest in ("/usr/local/bin/stockfish-pawn", "/tmp/stockfish-pawn", binaire):
        try:
            if dest != binaire:
                shutil.copy(binaire, dest)
            os.chmod(dest, 0o755)
            subprocess.run([dest, "quit"], check=True, timeout=10, capture_output=True)
            return dest
        except Exception as e:
            print("Stockfish ne démarre pas depuis", dest, ":", e)
    raise RuntimeError("Stockfish ne peut être exécuté nulle part")


def installer_stockfish(dossier=None):
    dossier = dossier or os.path.join(os.path.dirname(os.path.abspath(DOSSIER_POIDS)), "stockfish")
    deja = glob.glob(os.path.join(dossier, "**", "stockfish-ubuntu-*"), recursive=True)
    deja = [d for d in deja if os.path.isfile(d) and not d.endswith(".tar")]
    if deja:
        return rendre_executable(deja[0])
    os.makedirs(dossier, exist_ok=True)
    for url in URLS_STOCKFISH:
        try:
            tar = os.path.join(dossier, "sf.tar")
            urllib.request.urlretrieve(url, tar)
            with tarfile.open(tar) as t:
                t.extractall(dossier)
            binaire = [d for d in glob.glob(os.path.join(dossier, "**", "stockfish-ubuntu-*"), recursive=True)
                       if os.path.isfile(d) and not d.endswith(".tar")][0]
            return rendre_executable(binaire)
        except Exception as e:
            print("échec", url, e)
    if shutil.which("stockfish") is None:
        subprocess.run("apt-get -qq install -y stockfish", shell=True)
    for p in (shutil.which("stockfish"), "/usr/games/stockfish"):
        if p and os.path.exists(p):
            return p
    raise RuntimeError("impossible d'installer Stockfish")


# -- puzzles Lichess ---------------------------------------------------------------

def resoudre_puzzle(joueur, fen, moves):
    """format Lichess : FEN avant le coup adverse, moves[0] = coup adverse, puis solution.
    Comme sur Lichess, un mat est toujours accepté même si ce n'est pas le coup attendu.
    renvoie (résolu, premier_coup_bon, nb_coups_joués, secondes)"""
    b = chess.Board(fen)
    moves = moves.split()
    b.push_uci(moves[0])
    premier, joues, t0 = None, 0, time.perf_counter()
    for i in range(1, len(moves), 2):
        attendu = chess.Move.from_uci(moves[i])
        mv = coup(joueur, b)
        joues += 1
        b.push(mv)
        mat = b.is_checkmate()
        b.pop()
        ok = mv == attendu or mat
        if premier is None:
            premier = ok
        if not ok:
            return False, premier, joues, time.perf_counter() - t0
        if mat:
            break
        b.push(attendu)
        if i + 1 < len(moves):
            b.push_uci(moves[i + 1])
    return True, premier, joues, time.perf_counter() - t0


# -- parties -------------------------------------------------------------------------

def ecrire_suivi(suivi, blancs, noirs, b, ouverture, debut):
    # position en cours dans un petit fichier JSON, lu par le notebook pour l'afficher en direct
    dernier = b.peek() if b.move_stack else None
    san = ""
    if dernier:
        prec = b.copy()
        prec.pop()
        san = prec.san(dernier)
    tmp = suivi + ".tmp"
    with open(tmp, "w") as f:
        json.dump({"blancs": blancs, "noirs": noirs, "fen": b.fen(), "dernier": dernier.uci() if dernier else "",
                   "san": san, "ply": b.ply(), "ouverture": ouverture, "debut": debut}, f)
    os.replace(tmp, suivi)


def jouer_partie(blancs, noirs, ouverture, max_plies=300, suivi=None, nom_ouverture=""):
    """ouverture = liste de coups UCI joués d'office ; renvoie un dict (résultat, pgn, temps)"""
    import chess.pgn
    b = chess.Board()
    positions = []  # chaque coup : qui, coup, position obtenue, temps de réflexion
    for u in ouverture:
        mv = chess.Move.from_uci(u)
        positions.append({"ply": b.ply() + 1, "joueur": "ouverture", "san": b.san(mv), "uci": u, "secondes": 0.0})
        b.push(mv)
        positions[-1]["fen"] = b.fen()
    temps = {blancs: 0.0, noirs: 0.0}
    nb = {blancs: 0, noirs: 0}
    debut = time.time()
    if suivi:
        ecrire_suivi(suivi, blancs, noirs, b, nom_ouverture, debut)
    while not b.is_game_over(claim_draw=True) and b.ply() < max_plies:
        j = blancs if b.turn == chess.WHITE else noirs
        t0 = time.perf_counter()
        mv = coup(j, b)
        dt = time.perf_counter() - t0
        temps[j] += dt
        nb[j] += 1
        positions.append({"ply": b.ply() + 1, "joueur": j, "san": b.san(mv), "uci": mv.uci(), "secondes": round(dt, 3)})
        b.push(mv)
        positions[-1]["fen"] = b.fen()
        if suivi:
            ecrire_suivi(suivi, blancs, noirs, b, nom_ouverture, debut)
    res = b.result(claim_draw=True) if b.is_game_over(claim_draw=True) else "1/2-1/2"
    fin = b.outcome(claim_draw=True)
    jeu = chess.pgn.Game.from_board(b)
    jeu.headers.update(Event="Benchmark P.A.W.N.", White=blancs, Black=noirs, Result=res)
    return {
        "blancs": blancs, "noirs": noirs, "resultat": res,
        "fin": fin.termination.name if fin else "MAX_PLIES",
        "plies": b.ply(), "pgn": str(jeu),
        "s_par_coup_blancs": temps[blancs] / max(1, nb[blancs]),
        "s_par_coup_noirs": temps[noirs] / max(1, nb[noirs]),
        "positions": positions,
    }


# -- tâches pour ProcessPoolExecutor (doivent être importables depuis ce module) -------

def nb_gpu():
    try:
        r = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, timeout=10)
        return max(1, sum(l.startswith("GPU") for l in r.stdout.splitlines()))
    except Exception:
        return 1


def init_worker(gpu, threads, sf_chemin=None, sf_temps=0.1, gpu_mem_mo=None):
    # plusieurs processus x plusieurs réseaux sur un GPU : sans plafond, onnxruntime réserve
    # la mémoire trop largement et le T4 sature (BFCArena / CUBLAS_STATUS_ALLOC_FAILED).
    # Les processus sont répartis sur les GPU disponibles, avec un plafond par réseau.
    global GPU_MEM_MO, GPU_ID
    n = nb_gpu()
    GPU_ID = os.getpid() % n
    GPU_MEM_MO = gpu_mem_mo or (700 if n == 1 else 1200)
    configurer(gpu=gpu, threads=threads, sf_chemin=sf_chemin, sf_temps=sf_temps)


def tache_puzzle(args):
    joueur, pid, fen, moves = args
    ok, premier, joues, sec = resoudre_puzzle(joueur, fen, moves)
    return {"joueur": joueur, "PuzzleId": pid, "resolu": ok, "premier_coup": premier,
            "coups_joues": joues, "secondes": sec}


def tache_partie(args):
    blancs, noirs, ouverture, max_plies, id_ouv = args[:5]
    suivi = args[5] if len(args) > 5 else None   # dossier de suivi en direct (facultatif)
    fichier = None
    if suivi:
        os.makedirs(suivi, exist_ok=True)
        fichier = os.path.join(suivi, f"{blancs}_{noirs}_{id_ouv}.json".replace(" ", "-").replace("@", "-"))
    try:
        r = jouer_partie(blancs, noirs, ouverture, max_plies, fichier, id_ouv)
    finally:
        # un Stockfish ouvert garde un thread vivant qui empêche le processus fils de se terminer
        fermer_stockfish()
        if fichier and os.path.exists(fichier):
            os.remove(fichier)
    r["ouverture"] = id_ouv
    return r

In [ ]:
import os, sys, time, json, random
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import chess
sys.path.insert(0, os.getcwd())  # pour que les processus fils trouvent commun_echecs.py
import onnxruntime as ort
import commun_echecs as ce

print("onnxruntime", ort.__version__, "| providers dispo :", ort.get_available_providers())
ev_test = ce.Evaluateur("PAWN")
print("session PAWN sur :", ev_test.session.get_providers())
print("éval position initiale :", ev_test.eval1(chess.STARTING_FEN), "cp")

In [ ]:
SF = ce.installer_stockfish(os.path.join(os.getcwd(), "stockfish"))
print("Stockfish :", SF)

In [ ]:
# ---- réglages ----
JOUEUR = "PWN@ab3"
NIVEAUX = [1500, 1800, 2100, 2400, 2700]   # UCI_Elo de Stockfish (de 1320 à 3190)
SF_TEMPS = 0.1          # secondes par coup pour Stockfish
N_OUVERTURES = 8        # parties par niveau = 2 x N_OUVERTURES
MAX_PLIES = 200         # au-delà : nulle (évite les finales interminables)
NB_WORKERS = os.cpu_count()
BUDGET_H = 11           # on arrête de lancer des parties après ce temps (limite Kaggle : 12 h)
SEED = 0
SORTIE = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

OUVERTURES = {
    "Italienne": "e2e4 e7e5 g1f3 b8c6 f1c4 f8c5",
    "Espagnole": "e2e4 e7e5 g1f3 b8c6 f1b5 a7a6",
    "Sicilienne Najdorf": "e2e4 c7c5 g1f3 d7d6 d2d4 c5d4 f3d4 g8f6 b1c3 a7a6",
    "Française": "e2e4 e7e6 d2d4 d7d5 b1c3 g8f6",
    "Caro-Kann": "e2e4 c7c6 d2d4 d7d5 e4e5 c8f5",
    "Gambit dame refusé": "d2d4 d7d5 c2c4 e7e6 b1c3 g8f6",
    "Slave": "d2d4 d7d5 c2c4 c7c6 g1f3 g8f6",
    "Est-indienne": "d2d4 g8f6 c2c4 g7g6 b1c3 f8g7 e2e4 d7d6",
    "Nimzo-indienne": "d2d4 g8f6 c2c4 e7e6 b1c3 f8b4",
    "Anglaise": "c2c4 e7e5 b1c3 g8f6 g1f3 b8c6",
    "Scandinave": "e2e4 d7d5 e4d5 d8d5 b1c3 d5a5",
    "Londres": "d2d4 d7d5 g1f3 g8f6 c1f4 c7c5",
    "Pirc": "e2e4 d7d6 d2d4 g8f6 b1c3 g7g6",
    "Hollandaise": "d2d4 f7f5 g1f3 g8f6 g2g3 e7e6",
    "Écossaise": "e2e4 e7e5 g1f3 b8c6 d2d4 e5d4 f3d4 g8f6",
    "Réti": "g1f3 d7d5 g2g3 g8f6 f1g2 e7e6",
}
for nom, coups in OUVERTURES.items():  # vérifie que les ouvertures sont légales
    b = chess.Board()
    for u in coups.split():
        assert chess.Move.from_uci(u) in b.legal_moves, (nom, u)
        b.push_uci(u)
OUV = dict(list(OUVERTURES.items())[:N_OUVERTURES])

STOCKFISH = {f"SF@elo{e}": e for e in NIVEAUX}
taches = []
for sf in STOCKFISH:
    for nom, coups in OUV.items():
        taches.append((JOUEUR, sf, coups.split(), MAX_PLIES, nom))
        taches.append((sf, JOUEUR, coups.split(), MAX_PLIES, nom))
random.Random(SEED).shuffle(taches)  # si ça coupe, chaque niveau a à peu près le même nombre de parties
print(len(STOCKFISH), "niveaux,", len(taches), "parties")

## Parties (en parallèle, avec reprise)

In [ ]:
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed

PARTIES = f"{SORTIE}/pwn3_parties.jsonl"
DOSSIER = f"{SORTIE}/pwn3_parties"          # un sous-dossier par niveau, un fichier par partie
PAR_NIVEAU = 2 * len(OUV)
EMOJI = {"victoire": "✅", "nulle": "🤝", "défaite": "❌"}

def infos(r):
    pwn_blancs = r["blancs"] == JOUEUR
    adv = r["noirs"] if pwn_blancs else r["blancs"]
    couleur = "blancs" if pwn_blancs else "noirs"
    if r["resultat"] == "1/2-1/2":
        res = "nulle"
    else:
        res = "victoire" if (r["resultat"] == "1-0") == pwn_blancs else "défaite"
    return adv, couleur, res

def enregistrer(r):
    # fichiers lisibles pour chaque partie : PGN (à ouvrir sur lichess.org/paste) + CSV de toutes les positions
    adv, couleur, res = infos(r)
    d = os.path.join(DOSSIER, adv.replace("SF@", "stockfish_"))
    os.makedirs(d, exist_ok=True)
    nom = f"{r['ouverture']}_PWN-{couleur}".replace(" ", "-")
    with open(os.path.join(d, nom + ".pgn"), "w") as f:
        f.write(r["pgn"])
    pos = pd.DataFrame(r["positions"])
    pos.insert(0, "ouverture", r["ouverture"]); pos.insert(1, "PWN_joue", couleur); pos["resultat_PWN"] = res
    pos.to_csv(os.path.join(d, nom + "_positions.csv"), index=False)

def bilan_niveau(adv, lignes):
    res = [infos(r)[2] for r in lignes]
    v, n, p = res.count("victoire"), res.count("nulle"), res.count("défaite")
    print(f"\n🏁 NIVEAU {adv} TERMINÉ : {v} victoires, {n} nulles, {p} défaites → score {100 * (v + n / 2) / len(res):.0f} %")
    for r in sorted(lignes, key=lambda r: (r["ouverture"], infos(r)[1])):
        a, c, x = infos(r)
        print(f"   {EMOJI[x]} {r['ouverture']:20s} PWN avec les {c:6s} → {x:8s} ({r['fin']}, {r['plies'] // 2} coups)")
    print()

deja = []
if os.path.exists(PARTIES):
    deja = [json.loads(l) for l in open(PARTIES)]
fait = {(r["blancs"], r["noirs"], r["ouverture"]) for r in deja}
par_niveau = {sf: [r for r in deja if infos(r)[0] == sf] for sf in STOCKFISH}
reste = [t for t in taches if (t[0], t[1], t[4]) not in fait]
print(len(fait), "déjà jouées,", len(reste), "à jouer")
for sf, lignes in par_niveau.items():
    if len(lignes) >= PAR_NIVEAU:
        bilan_niveau(sf, lignes)

# ---- affichage en direct : une carte par partie en cours + tableau des scores ----
import glob, shutil
import chess.svg
from concurrent.futures import wait, FIRST_COMPLETED
from IPython.display import display, HTML

SUIVI = "/tmp/pwn3_en_cours"
shutil.rmtree(SUIVI, ignore_errors=True); os.makedirs(SUIVI)
derniers = []     # parties finies (HTML), les plus récentes à la fin
vu = {}           # partie -> dernier demi-coup affiché (pour n'animer que les nouveaux coups)
VALEUR = {chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3, chess.ROOK: 5, chess.QUEEN: 9}
SYMB = {chess.WHITE: {chess.PAWN: "♙", chess.KNIGHT: "♘", chess.BISHOP: "♗", chess.ROOK: "♖", chess.QUEEN: "♕"},
        chess.BLACK: {chess.PAWN: "♟", chess.KNIGHT: "♞", chess.BISHOP: "♝", chess.ROOK: "♜", chess.QUEEN: "♛"}}
DEPART = {chess.PAWN: 8, chess.KNIGHT: 2, chess.BISHOP: 2, chess.ROOK: 2, chess.QUEEN: 1}
COULEURS = {"square light": "#eeeed2", "square dark": "#769656",
            "square light lastmove": "#f6f669", "square dark lastmove": "#baca2b", "margin": "#312e2b", "coord": "#ddd"}
TAILLE = 45   # taille d'une case dans le repère SVG de chess.svg (marge 15)

STYLE = """<style>
.pw{font-family:system-ui,sans-serif;color:#222;max-width:1250px}
.pw h2{margin:4px 0 2px}.pw .aide{background:#f4f6f8;border-radius:8px;padding:8px 12px;font-size:13px;margin:8px 0}
.pw .niv{display:flex;gap:10px;flex-wrap:wrap;margin:10px 0}
.pw .nv{border:1px solid #ddd;border-radius:10px;padding:8px 10px;width:190px;background:#fff;font-size:13px}
.pw .nv.fini{border-color:#2e7d32;background:#eef7ee}
.pw .barre{display:flex;height:10px;border-radius:5px;overflow:hidden;background:#eee;margin:5px 0}
.pw .cartes{display:flex;flex-wrap:wrap;gap:12px}
.pw .carte{border:1px solid #ccc;border-radius:12px;padding:8px;background:#fff;width:250px;box-shadow:0 1px 4px #0001}
.pw .titre{font-size:13px;color:#555;margin-bottom:6px}
.pw .joueur{display:flex;align-items:center;justify-content:space-between;padding:4px 6px;border-radius:6px;font-size:13px;white-space:nowrap}
.pw .joueur.pwn{background:#e3f0ff}.pw .joueur.sf{background:#f3f3f3}
.pw .pion{display:inline-block;width:13px;height:13px;border-radius:50%;border:1px solid #333;margin-right:6px;vertical-align:-2px}
.pw .reflechit{color:#d35400;font-weight:bold;animation:clig 1s infinite}
@keyframes clig{50%{opacity:.25}}
.pw .pris{font-size:15px;letter-spacing:-2px;color:#555;min-height:18px}
.pw .info{font-size:13px;margin-top:6px}
.pw .mat{height:8px;border-radius:4px;background:#ddd;position:relative;margin-top:4px}
.pw .fin{font-size:13px;padding:3px 0;border-bottom:1px solid #eee}
</style>"""

def case_xy(sq, retourne):
    f, r = chess.square_file(sq), chess.square_rank(sq)
    if retourne:
        return 15 + TAILLE * (7 - f), 15 + TAILLE * r
    return 15 + TAILLE * f, 15 + TAILLE * (7 - r)

def echiquier(bd, coup, retourne, animer):
    svg = chess.svg.board(bd, lastmove=coup, size=250, flipped=retourne, colors=COULEURS,
                          check=bd.king(bd.turn) if bd.is_check() else None)
    if coup and animer:   # la pièce glisse de sa case de départ à sa case d'arrivée
        xa, ya = case_xy(coup.from_square, retourne)
        xb, yb = case_xy(coup.to_square, retourne)
        cible = f'transform="translate({xb}, {yb})" />'
        anim = (f'transform="translate({xb}, {yb})"><animateTransform attributeName="transform" type="translate" '
                f'from="{xa} {ya}" to="{xb} {yb}" dur="0.7s" fill="freeze" /></use>')
        svg = svg.replace(cible, anim, 1)
    return svg

def pieces_prises(bd, couleur):
    # pièces de `couleur` qui ont disparu de l'échiquier
    return "".join(SYMB[couleur][p] * max(0, DEPART[p] - len(bd.pieces(p, couleur))) for p in DEPART)

def materiel(bd, couleur):
    return sum(v * len(bd.pieces(p, couleur)) for p, v in VALEUR.items())

def barre_joueur(nom, classe, couleur_pieces, a_son_tour, prises_adverses):
    pastille = "#fff" if couleur_pieces == chess.WHITE else "#222"
    txt_couleur = "blancs" if couleur_pieces == chess.WHITE else "noirs"
    etat = "<span class='reflechit'>⏳ joue…</span>" if a_son_tour else ""
    return (f"<div class='joueur {classe}'><span><span class='pion' style='background:{pastille}'></span>"
            f"<b>{nom}</b> <small>({txt_couleur})</small></span>{etat}</div>"
            f"<div class='pris' title='pièces capturées'>{prises_adverses}</div>")

def carte(e):
    pwn_blancs = e["blancs"] == JOUEUR
    adv = e["noirs"] if pwn_blancs else e["blancs"]
    c_pwn = chess.WHITE if pwn_blancs else chess.BLACK
    bd = chess.Board(e["fen"])
    coup = chess.Move.from_uci(e["dernier"]) if e["dernier"] else None
    cle = (e["blancs"], e["noirs"], e["ouverture"])
    animer = vu.get(cle) != e["ply"]
    vu[cle] = e["ply"]
    ecart = materiel(bd, c_pwn) - materiel(bd, not c_pwn)
    if ecart > 0:
        mat_txt, mat_coul = f"PWN mène de <b>+{ecart}</b>", "#1e88e5"
    elif ecart < 0:
        mat_txt, mat_coul = f"Stockfish mène de <b>+{-ecart}</b>", "#e53935"
    else:
        mat_txt, mat_coul = "matériel <b>égal</b>", "#999"
    pos = 50 + max(-50, min(50, ecart * 5))
    num = (e["ply"] + 1) // 2
    if coup:
        qui = "PWN" if (e["ply"] % 2 == 1) == pwn_blancs else "Stockfish"
        dernier = f"{num}{'.' if e['ply'] % 2 else '…'} <b>{e['san']}</b> ({qui})"
    else:
        dernier = "–"
    duree = (time.time() - e["debut"]) / 60
    haut = barre_joueur(f"Stockfish {adv[6:]}", "sf", not c_pwn, bd.turn != c_pwn, pieces_prises(bd, c_pwn))
    bas = barre_joueur(JOUEUR, "pwn", c_pwn, bd.turn == c_pwn, pieces_prises(bd, not c_pwn))
    return (f"<div class='carte'><div class='titre'>📖 {e['ouverture']} · coup {num} · {duree:.0f} min</div>"
            f"{haut}{echiquier(bd, coup, not pwn_blancs, animer)}{bas}"
            f"<div class='info'>Dernier coup : {dernier}</div>"
            f"<div class='info'>⚖️ {mat_txt}</div>"
            f"<div class='mat'><div style='position:absolute;left:{pos}%;top:-3px;width:4px;height:14px;background:{mat_coul}'></div>"
            f"<div style='position:absolute;left:50%;top:0;width:1px;height:8px;background:#555'></div></div></div>")

def ecran():
    fait_n = sum(len(v) for v in par_niveau.values())
    total = PAR_NIVEAU * len(STOCKFISH)
    en_cours = []
    for fich in sorted(glob.glob(f"{SUIVI}/*.json")):
        try:
            e = json.load(open(fich))
        except Exception:
            continue
        if "ouverture" in e:   # sinon : partie lancée par une ancienne version du notebook
            en_cours.append(e)
    h = [STYLE, "<div class='pw'>",
         f"<h2>♟️ {JOUEUR} contre Stockfish 17</h2>",
         f"<div>{fait_n}/{total} parties terminées · {len(en_cours)} en cours · {(time.time() - t0) / 60:.0f} min</div>",
         f"<div class='barre' style='width:600px'><div style='width:{100 * fait_n / total:.1f}%;background:#1e88e5'></div></div>",
         "<div class='aide'>Chaque carte = une partie en cours. <b style='color:#1e88e5'>PWN</b> (ton IA) est toujours "
         "<b>en bas</b>, <b>Stockfish</b> en haut. La pastille ⚪/⚫ indique la couleur des pièces. Les cases jaunes = dernier coup. "
         "Sous chaque joueur : les pièces qu'il a capturées. Les niveaux sont joués dans un ordre mélangé exprès.</div>",
         "<div class='niv'>"]
    for sf, lignes in par_niveau.items():
        res = [infos(r)[2] for r in lignes]
        v, n, d = res.count("victoire"), res.count("nulle"), res.count("défaite")
        cours = sum(1 for e in en_cours if sf in (e["blancs"], e["noirs"]))
        sc = f"{100 * (v + n / 2) / len(res):.0f} %" if res else "–"
        fini = " fini" if len(res) == PAR_NIVEAU else ""
        w = lambda x: 100 * x / PAR_NIVEAU
        h.append(f"<div class='nv{fini}'><b>Stockfish {sf[6:]}</b>{' ✔️' if fini else ''}"
                 f"<div class='barre'><div style='width:{w(v)}%;background:#43a047'></div>"
                 f"<div style='width:{w(n)}%;background:#fbc02d'></div><div style='width:{w(d)}%;background:#e53935'></div></div>"
                 f"<small>✅ {v} · 🤝 {n} · ❌ {d} · {len(res)}/{PAR_NIVEAU}"
                 f"{f' · {cours} en cours' if cours else ''}</small><br>Score PWN : <b>{sc}</b></div>")
    h.append("</div><div class='cartes'>")
    for e in en_cours:
        try:
            h.append(carte(e))
        except Exception:
            pass   # fichier d'une ancienne partie (ex. kernel pas redémarré) : on l'ignore
    h.append("</div><h3>Dernières parties terminées</h3>")
    h.append("".join(derniers[-10:][::-1]) or "<i>aucune pour l'instant (une partie dure 10–20 min)</i>")
    h.append("</div>")
    return HTML("".join(h))

def ligne_fin(r):
    adv, couleur, res = infos(r)
    coul = {"victoire": "#43a047", "nulle": "#f9a825", "défaite": "#e53935"}[res]
    return (f"<div class='fin'>{EMOJI[res]} <b style='color:{coul}'>{res.upper()}</b> de PWN ({couleur}) contre "
            f"<b>Stockfish {adv[6:]}</b> · {r['ouverture']} · {r['plies'] // 2} coups · {r['fin'].lower()}</div>")

t0 = time.time()
ecran_id = display(ecran(), display_id=True)
ex = ProcessPoolExecutor(NB_WORKERS, mp_context=mp.get_context("spawn"),
                         initializer=ce.init_worker, initargs=(False, 1, SF, SF_TEMPS))
en_attente = {ex.submit(ce.tache_partie, t + (SUIVI,)) for t in reste}
k = 0
with open(PARTIES, "a") as f:
    while en_attente:
        finis, en_attente = wait(en_attente, timeout=2, return_when=FIRST_COMPLETED)
        for fu in finis:
            k += 1
            try:
                r = fu.result()
            except Exception as e:
                print("erreur :", e)
                continue
            f.write(json.dumps(r) + "\n"); f.flush()
            enregistrer(r)
            adv, couleur, res = infos(r)
            par_niveau[adv].append(r)
            ligne = (f"{EMOJI[res]} contre {adv} ({len(par_niveau[adv])}/{PAR_NIVEAU}) : {r['ouverture']}, "
                     f"PWN avec les {couleur} → {res} ({r['fin']}, {r['plies'] // 2} coups)")
            derniers.append(ligne_fin(r))
            print(f"[{k}/{len(reste)} | {(time.time() - t0) / 60:.0f} min] {ligne}")
            if len(par_niveau[adv]) == PAR_NIVEAU:
                bilan_niveau(adv, par_niveau[adv])
        ecran_id.update(ecran())
        if time.time() - t0 > BUDGET_H * 3600:
            print("budget temps atteint, on s'arrête là")
            for x in en_attente:
                x.cancel()
            break
ex.shutdown(wait=False, cancel_futures=True)

## Résultats

In [ ]:
parties = pd.DataFrame([json.loads(l) for l in open(PARTIES)])
parties = parties.drop_duplicates(["blancs", "noirs", "ouverture"])
parties = parties[parties.blancs.isin(STOCKFISH) | parties.noirs.isin(STOCKFISH)]
parties["adversaire"] = np.where(parties.blancs == JOUEUR, parties.noirs, parties.blancs)
parties["couleur"] = np.where(parties.blancs == JOUEUR, "blancs", "noirs")
gagne = {"1-0": "blancs", "0-1": "noirs"}
parties["resultat_pwn"] = [
    "nulle" if r == "1/2-1/2" else ("victoire" if gagne[r] == c else "défaite")
    for r, c in zip(parties.resultat, parties.couleur)]
parties["score"] = parties.resultat_pwn.map({"victoire": 1.0, "nulle": 0.5, "défaite": 0.0})
with open(f"{SORTIE}/pwn3_parties.pgn", "w") as f:
    f.write("\n\n".join(parties.pgn))
toutes = []
for _, r in parties.iterrows():
    pos = pd.DataFrame(r.positions)
    pos.insert(0, "adversaire", r.adversaire); pos.insert(1, "ouverture", r.ouverture)
    pos.insert(2, "PWN_joue", r.couleur); pos["resultat_PWN"] = r.resultat_pwn
    toutes.append(pos)
pd.concat(toutes).to_csv(f"{SORTIE}/pwn3_positions.csv", index=False)
liste = parties[["adversaire", "ouverture", "couleur", "resultat_pwn", "fin", "plies"]].copy()
liste["coups"] = liste.pop("plies") // 2
liste.sort_values(["adversaire", "ouverture", "couleur"]).to_csv(f"{SORTIE}/pwn3_liste_parties.csv", index=False)

tab = parties.groupby("adversaire").agg(
    victoires=("resultat_pwn", lambda s: (s == "victoire").sum()),
    nulles=("resultat_pwn", lambda s: (s == "nulle").sum()),
    defaites=("resultat_pwn", lambda s: (s == "défaite").sum()),
    parties=("score", "size"),
    score_pct=("score", lambda s: 100 * s.mean()),
    score_blancs=("score", lambda s: 100 * s[parties.loc[s.index, "couleur"] == "blancs"].mean()),
    score_noirs=("score", lambda s: 100 * s[parties.loc[s.index, "couleur"] == "noirs"].mean()),
)
tab["Elo Stockfish"] = tab.index.map(STOCKFISH)
tab = tab.sort_values("Elo Stockfish")
print(len(parties), "parties | fins :", parties.fin.value_counts().to_dict())
print("temps moyen par coup de", JOUEUR, ":",
      round(pd.concat([parties[parties.blancs == JOUEUR].s_par_coup_blancs,
                       parties[parties.noirs == JOUEUR].s_par_coup_noirs]).mean(), 2), "s")
tab.round(1)

In [ ]:
from scipy.optimize import minimize_scalar

def elo_mle(df):
    """Elo de PWN par maximum de vraisemblance, les niveaux Stockfish étant fixés (nulle = ½ victoire)"""
    e_adv, s = df.adversaire.map(STOCKFISH).to_numpy(), df.score.to_numpy()
    def nll(r):
        p = np.clip(1 / (1 + 10 ** ((e_adv - r) / 400)), 1e-9, 1 - 1e-9)
        return -np.sum(s * np.log(p) + (1 - s) * np.log(1 - p))
    return minimize_scalar(nll, bounds=(0, 4000), method="bounded").x

elo = elo_mle(parties)
rng = np.random.default_rng(SEED)
boot = [elo_mle(parties.sample(len(parties), replace=True, random_state=int(rng.integers(1e9)))) for _ in range(500)]
bas, haut = np.percentile(boot, [2.5, 97.5])
print(f"Elo de {JOUEUR} en partie : {elo:.0f}  (IC 95 % : {bas:.0f} – {haut:.0f})")
pd.DataFrame([{"joueur": JOUEUR, "Elo": elo, "IC 95 % bas": bas, "IC 95 % haut": haut, "parties": len(parties)}]) \
  .to_csv(f"{SORTIE}/pwn3_elo.csv", index=False)
tab.to_csv(f"{SORTIE}/pwn3_par_niveau.csv")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.linspace(min(NIVEAUX) - 100, max(NIVEAUX) + 100, 200)
ax.plot(x, 100 / (1 + 10 ** ((x - elo) / 400)), color="gray", lw=1, ls="--", label=f"attendu pour Elo {elo:.0f}")
ax.bar(tab["Elo Stockfish"], tab.score_pct, width=120, color="#58a", label="score réel de " + JOUEUR)
for e, v, n in zip(tab["Elo Stockfish"], tab.score_pct, tab.parties):
    ax.text(e, v + 2, f"{v:.0f} %", ha="center", fontsize=9)
ax.axhline(50, color="black", lw=0.5)
ax.axvline(elo, color="#c55", lw=1.5, label=f"Elo estimé {elo:.0f} [{bas:.0f}–{haut:.0f}]")
ax.set_xticks(NIVEAUX); ax.set_xlabel("niveau de Stockfish (UCI_Elo)"); ax.set_ylabel("score de PWN@ab3 (%)")
ax.set_ylim(0, 110); ax.set_title(f"{JOUEUR} contre Stockfish 17"); ax.legend(loc="upper right", fontsize=8)
plt.tight_layout(); plt.savefig(f"{SORTIE}/pwn3_vs_stockfish.png", dpi=120); plt.show()

### Les fichiers (onglet Output)
| Fichier | Contenu |
|---|---|
| `pwn3_parties/stockfish_eloXXXX/<ouverture>_PWN-<couleur>.pgn` | chaque partie, à coller sur **lichess.org/paste** pour la rejouer |
| `pwn3_parties/stockfish_eloXXXX/<ouverture>_PWN-<couleur>_positions.csv` | chaque coup de la partie : qui joue, le coup, la **position (FEN)** obtenue, le temps de réflexion |
| `pwn3_positions.csv` | toutes les positions de toutes les parties dans un seul fichier |
| `pwn3_liste_parties.csv` | la liste des parties : adversaire, ouverture, couleur de PWN, résultat, comment ça finit, nombre de coups |
| `pwn3_par_niveau.csv`, `pwn3_elo.csv`, `pwn3_vs_stockfish.png`, `pwn3_parties.pgn` | les bilans, l'Elo, la courbe, toutes les parties en un PGN |

### Comment lire les résultats
- Le niveau où le score passe sous **50 %**, c'est à peu près l'Elo de PWN@ab3.
- Si PWN fait encore > 50 % contre le niveau le plus fort, ajoute des niveaux plus hauts dans `NIVEAUX` (ex. 2700, 2900) et relance :
  les parties déjà jouées sont gardées.
- Avec 16 parties par niveau (80 au total), l'intervalle fait environ ±70–120 Elo. Pour plus de précision : `N_OUVERTURES = 12` (plus long).
- Beaucoup de fins `MAX_PLIES` ou de nulles par répétition → PWN ne sait pas convertir un avantage en finale.